# Fine Tuned Model

Using twitter-roberta-base-sentiment as a base, fine tunes a model to predict whether a reply is in response to a liberal, conservative, or neutral account name

In [2]:
!pip install pyprojroot
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from transformers import AutoTokenizer, AutoConfig
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import load_dataset, Features, Value, ClassLabel

import polars as pl
import numpy as np
import time
import evaluate

from pyprojroot import here
from scipy.special import softmax
from ast import literal_eval

In [5]:
# Setup model and tokenizer
MODEL = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

In [13]:
# Load train and test data
dataset = load_dataset("parquet", data_files={
    'train': "drive/MyDrive/Data/train_reduced.parquet",
    'test': "drive/MyDrive/Data/test_reduced.parquet"})
dataset = dataset.class_encode_column("label")

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Stringifying the column:   0%|          | 0/48678 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/48678 [00:00<?, ? examples/s]

Stringifying the column:   0%|          | 0/12406 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/12406 [00:00<?, ? examples/s]

In [14]:
dataset

DatasetDict({
    train: Dataset({
        features: ['index', 'id', 'subreddit', 'username', 'username_score', 'content', 'label'],
        num_rows: 48678
    })
    test: Dataset({
        features: ['index', 'id', 'subreddit', 'username', 'username_score', 'content', 'label'],
        num_rows: 12406
    })
})

In [42]:
# Preprocess text (username and link placeholders)
def preprocess_and_tokenize(data):
    cleaned_text = []
    for text in data["content"]:
      new_text = []
      for t in text.split():
          t = '@user' if t.startswith('@') and len(t) > 1 else t
          t = 'http' if t.startswith('http') else t
          new_text.append(t)
      cleaned_text.append(" ".join(new_text))

    return tokenizer(cleaned_text, return_tensors='pt', padding=True, truncation=True, max_length=512)

def inference_all(encoded_input, trainer) -> dict[str, np.float32]:
    output = trainer.predict(encoded_input)
    scores = softmax(output.predictions, axis=1)
    return [{label: float(confidence) for label, confidence
             in zip(dataset["test"].features["label"].names, score)
             } for score in scores]

In [16]:
train_encoded = dataset["train"].map(preprocess_and_tokenize, batched=True)
test_encoded = dataset["test"].map(preprocess_and_tokenize, batched=True)

Map:   0%|          | 0/48678 [00:00<?, ? examples/s]

Map:   0%|          | 0/12406 [00:00<?, ? examples/s]

In [10]:
for name, _ in model.named_parameters():
    print(name)

roberta.embeddings.word_embeddings.weight
roberta.embeddings.token_type_embeddings.weight
roberta.embeddings.LayerNorm.weight
roberta.embeddings.LayerNorm.bias
roberta.embeddings.position_embeddings.weight
roberta.encoder.layer.0.attention.self.query.weight
roberta.encoder.layer.0.attention.self.query.bias
roberta.encoder.layer.0.attention.self.key.weight
roberta.encoder.layer.0.attention.self.key.bias
roberta.encoder.layer.0.attention.self.value.weight
roberta.encoder.layer.0.attention.self.value.bias
roberta.encoder.layer.0.attention.output.dense.weight
roberta.encoder.layer.0.attention.output.dense.bias
roberta.encoder.layer.0.attention.output.LayerNorm.weight
roberta.encoder.layer.0.attention.output.LayerNorm.bias
roberta.encoder.layer.0.intermediate.dense.weight
roberta.encoder.layer.0.intermediate.dense.bias
roberta.encoder.layer.0.output.dense.weight
roberta.encoder.layer.0.output.dense.bias
roberta.encoder.layer.0.output.LayerNorm.weight
roberta.encoder.layer.0.output.LayerNorm

In [11]:
metric = evaluate.load('f1')

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels, average='macro')

In [ ]:
batch_size = 32
num_epochs = 3

model_name = f"fine_tune_batch{batch_size}_epochs{num_epochs}_train_classifier_11_embed"

layers_to_train = ["classfier", "layer.11", "embeddings"]
for name, param in model.named_parameters():
    param.requires_grad = any(x in name for x in layers_to_train)

training_args = TrainingArguments(
    output_dir=f"drive/MyDrive/Data/{model_name}",
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to='none'
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_encoded,
    eval_dataset=test_encoded,
    compute_metrics=compute_metrics
)
trainer.train()

Epoch,Training Loss,Validation Loss


In [24]:
trainer.save_model(f"drive/MyDrive/Data/{model_name}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [25]:
test_out = trainer.predict(test_encoded)

In [27]:
test_out

PredictionOutput(predictions=array([[ 0.15060183, -0.9265235 ,  0.386463  ],
       [-1.0176982 ,  0.13332176,  0.40947455],
       [ 0.13472797, -2.271765  ,  1.7656095 ],
       ...,
       [ 1.3492801 , -0.6914794 , -1.5417715 ],
       [ 3.0041337 , -2.7474802 , -0.58128285],
       [-0.7028041 , -1.5751686 ,  2.105443  ]], dtype=float32), label_ids=array([0, 2, 2, ..., 2, 0, 1]), metrics={'test_loss': 1.3431167602539062, 'test_f1': 0.4794123134311942, 'test_runtime': 348.8318, 'test_samples_per_second': 35.564, 'test_steps_per_second': 2.225})

In [33]:
scores = softmax(test_out.predictions, axis=1)

In [37]:
dataset["test"].features["label"].names

['conversative', 'liberal', 'neutral']

In [43]:
train_scores = inference_all(train_encoded, trainer)
test_scores = inference_all(test_encoded, trainer)

In [77]:
train_df = dataset["train"].to_polars()
test_df = dataset["test"].to_polars()

In [78]:
test_df.head(10)

index,id,subreddit,username,username_score,content,label
u32,str,str,str,f64,str,i64
11508213,"""d0oshg2""","""Fitness""","""Magahaka""",0.95,"""""Eat your regular meals that y…",0
8398748,"""caphney""","""gardening""","""louchar""",0.0,"""'I used to get baked and do th…",2
28204105,"""cwufxng""","""Fantasy""","""xolsiion""",0.0,"""""I've read both Riyira Chronic…",2
7887909,"""e22e4z8""","""movies""","""LivnLegndNeedsEggs""",-0.7,"""'I second this. It is so obno…",1
2411655,"""cjblu8y""","""movies""","""RandomJPG6""",0.0,"""'R \n18+ \nM \nWhatever reg…",2
29022391,"""co9jeo6""","""CasualConversation""","""Wooble_Gop""",0.8,"""""I feel this way when I'm stud…",0
35839305,"""caarcbr""","""photography""","""HoshPoshMosh""",0.0,"""""I've never used Hugin. How do…",2
12855341,"""cht8ktz""","""MechanicAdvice""","""IhateFoxnews""",0.65,"""""It sounds like the coil pack …",0
16283734,"""deuctoh""","""explainlikeimfive""","""harley4570""",0.0,"""'[removed]'""",2


In [86]:
train_df = train_df.drop(pl.col("index"))
test_df = test_df.drop(pl.col("index"))

In [80]:
label_map = {id: name for id, name in enumerate(dataset["test"].features["label"].names)}
train_df = train_df.with_columns(pl.col("label").replace_strict(label_map, return_dtype=pl.String).alias("label"))
test_df = test_df.with_columns(pl.col("label").replace_strict(label_map, return_dtype=pl.String).alias("label"))

In [88]:
train_df = train_df.with_columns(pl.Series("score", train_scores)).unnest("score")
test_df = test_df.with_columns(pl.Series("score", test_scores)).unnest("score")

In [89]:
test_df.head()

id,subreddit,username,username_score,content,label,conversative,liberal,neutral
str,str,str,f64,str,str,f64,f64,f64
"""d0oshg2""","""Fitness""","""Magahaka""",0.95,"""""Eat your regular meals that y…","""conversative""",0.383646,0.130659,0.485695
"""caphney""","""gardening""","""louchar""",0.0,"""'I used to get baked and do th…","""neutral""",0.120072,0.379598,0.500329
"""cwufxng""","""Fantasy""","""xolsiion""",0.0,"""""I've read both Riyira Chronic…","""neutral""",0.161329,0.014541,0.82413
"""e22e4z8""","""movies""","""LivnLegndNeedsEggs""",-0.7,"""'I second this. It is so obno…","""liberal""",0.001088,0.997398,0.001514
"""cjblu8y""","""movies""","""RandomJPG6""",0.0,"""'R \n18+ \nM \nWhatever reg…","""neutral""",0.007709,0.005113,0.987177


In [92]:
calc_prediction = pl.when(pl.col("liberal") > pl.col("neutral")).then(
        pl.when(pl.col("liberal") > pl.col("conversative"))
        .then(pl.lit("liberal"))
        .otherwise(pl.lit("conversative"))
    ).otherwise(pl.when(pl.col("neutral") > pl.col("conversative"))
        .then(pl.lit("neutral"))
        .otherwise(pl.lit("conversative"))
    ).alias("prediction")

train_df = train_df.with_columns(calc_prediction)
test_df = test_df.with_columns(calc_prediction)

In [99]:
train_cm = train_df.group_by(["label", "prediction"]).agg(pl.len().alias("count"))
test_cm = test_df.group_by(["label", "prediction"]).agg(pl.len().alias("count"))


In [100]:
test_cm

label,prediction,count
str,str,u32
"""neutral""","""neutral""",4104
"""conversative""","""liberal""",290
"""neutral""","""liberal""",630
"""conversative""","""conversative""",1675
"""liberal""","""conversative""",558
"""liberal""","""liberal""",771
"""neutral""","""conversative""",1469
"""liberal""","""neutral""",1310
"""conversative""","""neutral""",1599


In [105]:
def get_tp(cm, class_name):
  return cm.filter((pl.col("label") == class_name) & (pl.col("prediction") == class_name)).get_column("count").item()

def get_fp(cm, class_name):
  return cm.filter((pl.col("label") != class_name) & (pl.col("prediction") == class_name)).get_column("count").sum()

def get_fn(cm, class_name):
  return cm.filter((pl.col("label") == class_name) & (pl.col("prediction") != class_name)).get_column("count").sum()

def get_precision(cm, class_name):
  return get_tp(cm, class_name) / (get_tp(cm, class_name) + get_fp(cm, class_name))

def get_recall(cm, class_name):
  return get_tp(cm, class_name) / (get_tp(cm, class_name) + get_fn(cm, class_name))

def get_f1(cm, class_name):
  precision = get_precision(cm, class_name)
  recall = get_recall(cm, class_name)
  return 2 * (precision * recall) / (precision + recall)

In [111]:
precision_liberal = get_precision(test_cm, "liberal")
recall_liberal = get_recall(test_cm, "liberal")
f1_liberal = get_f1(test_cm, "liberal")

precision_neutral = get_precision(test_cm, "neutral")
recall_neutral = get_recall(test_cm, "neutral")
f1_neutral = get_f1(test_cm, "neutral")

precision_conversative = get_precision(test_cm, "conversative")
recall_conversative = get_recall(test_cm, "conversative")
f1_conversative = get_f1(test_cm, "conversative")

print("Liberal")
print(f"Precision: {precision_liberal}")
print(f"Recall: {recall_liberal}")
print(f"F1: {f1_liberal}")
print()

print("Neutral")
print(f"Precision: {precision_neutral}")
print(f"Recall: {recall_neutral}")
print(f"F1: {f1_neutral}")
print()

print("Conversative")
print(f"Precision: {precision_conversative}")
print(f"Recall: {recall_conversative}")
print(f"F1: {f1_conversative}")



Liberal
Precision: 0.4559432288586635
Recall: 0.29215611974232664
F1: 0.35612009237875286

Neutral
Precision: 0.5851989162983031
Recall: 0.6616153474125424
F1: 0.6210653753026635

Conversative
Precision: 0.45245813074014046
Recall: 0.46997755331088664
F1: 0.4610514726121663


In [112]:
train_df.write_parquet(f"drive/MyDrive/Data/train_reduced_{model_name}.parquet")
test_df.write_parquet(f"drive/MyDrive/Data/test_reduced_{model_name}.parquet")